# Baseline: Jev vs Claude Opus vs Claude Sonnet on one cluster

Two marker lists, same question set, three models. Only two things are compared: **how long each call takes** and **what each model answers**.

| Case | `marker_genes` | `tissue` |
|---|---|---|
| `base` | CD3D, CD3E, TRAC, IL7R, LTB | blood |
| `base + LST1` | the same five plus **LST1** (a monocyte gene) appended last | blood |

The state and questions are the default [JevCellType](https://github.com/yw-Hua/JevCellType) payload, stored in `inputs/`:
- `broad_type`: a choice among 8 options;
- `cell_type`: a choice among 26 options;
- `identity_coherence`: a 0–4 score.

**How each model is asked**
- **Jev** receives the payload exactly as written in `inputs/`.
- **Claude** receives the same state and the same questions as JSON. Its answer is constrained by a JSON schema: each choice must be one of the option keys, and coherence must be a level from 0 to 4.

**Notes on the comparison**
- **Time** is wall-clock seconds per call, network included. Calls run one at a time, never in parallel.
- **Retries**: a call is retried only on rate limits, overload, or network errors. Only the successful attempt is timed.
- **Coherence scales differ**: Jev returns the probability-weighted level (e.g. 3.67); Claude returns one whole level.
- **Model pinning**: Claude runs without server-side refusal fallbacks, so every answer comes from the model named in its row.
- **Cost**: these calls are real and billed (pennies in total). Jev's per-call cost is far below Claude's.

## Setup

Run this notebook from `Tests/Baseline/`.

**Keys** are never stored in this repo. Each key is read from an environment variable first, then from a one-line file in the repo-level `keys/` folder (which is git-ignored):
- **Jev**: `TYPESAFE_API_KEY` (TypeSafe API) **or** `OPENROUTER_API_KEY` (OpenRouter Decisions endpoint).
- **Claude**: `ANTHROPIC_API_KEY`, or a profile from `ant auth login`. The Anthropic SDK resolves this on its own.

In [ ]:
import json
import os
import time
from collections import Counter
from pathlib import Path

import anthropic
import pandas as pd
import requests

HERE = Path.cwd()                      # run from Tests/Baseline/
INPUTS, OUTPUTS = HERE / "inputs", HERE / "outputs"
KEYS = HERE.parents[1] / "keys"        # <repo>/keys, git-ignored

N_REPEATS = 3                          # identical calls per model per case
MAX_ATTEMPTS = 3                       # retries only for rate limits, overload, network

CLAUDE_MODELS = {"Claude Opus 5.5": "claude-opus-5-5", "Claude Sonnet 5.5": "claude-sonnet-5-5"}
CLAUDE_EFFORT = "medium"               # same for both; defaults differ (Opus 5.5: medium, Sonnet 5.5: high)

JEV_ROUTES = {
    "typesafe":   {"endpoint": "https://api.typesafe.ai/v1/systemone",      "model": "jev-1.13.0",        "key": "TYPESAFE_API_KEY"},
    "openrouter": {"endpoint": "https://openrouter.ai/api/alpha/decisions", "model": "typesafe/jev-1.13", "key": "OPENROUTER_API_KEY"},
}
JEV_ROUTE = None                       # None: use the first route whose key is found

## Inputs: the two cases

In [ ]:
state = json.loads((INPUTS / "state.json").read_text())
questions = json.loads((INPUTS / "questions.json").read_text())

cases = {
    "base": state,
    "base + LST1": {**state, "marker_genes": state["marker_genes"] + ["LST1"]},
}
for name, s in cases.items():
    print(f"{name:12s} {s}")
for q, spec in questions.items():
    print(f"{q:20s} {spec['type']:6s} {len(spec['criteria'])} options")

## Callers

Each caller returns `(seconds, answer, raw_response)`. `timed()` retries transient failures and keeps the timing of the successful attempt only.

In [ ]:
class Transient(Exception):
    """A failure worth retrying: rate limit, overload, or network."""


def read_key(name):
    """Environment variable first, then a one-line file keys/<name>."""
    value = os.environ.get(name, "").strip()
    if not value and (KEYS / name).is_file():
        value = (KEYS / name).read_text(encoding="utf-8").strip()
    return value or None


def timed(call):
    for attempt in range(1, MAX_ATTEMPTS + 1):
        try:
            seconds, answer, raw = call()
            return {"status": "ok", "attempts": attempt, "seconds": round(seconds, 3), **answer}, raw
        except Transient as exc:
            last_error = str(exc)
            time.sleep(2 ** attempt)
    return {"status": "error", "attempts": MAX_ATTEMPTS, "error": last_error}, None

In [ ]:
def find_jev_route():
    for name in [JEV_ROUTE] if JEV_ROUTE else list(JEV_ROUTES):
        key = read_key(JEV_ROUTES[name]["key"])
        if key:
            return name, key
    raise RuntimeError("No Jev key found: set TYPESAFE_API_KEY or OPENROUTER_API_KEY, "
                       "or save the key alone in keys/TYPESAFE_API_KEY or keys/OPENROUTER_API_KEY.")


def ask_jev(state, session, route, key):
    cfg = JEV_ROUTES[route]
    payload = {"model": cfg["model"], "state": state, "questions": questions}
    t0 = time.perf_counter()
    try:
        r = session.post(cfg["endpoint"], json=payload, timeout=60,
                         headers={"Authorization": f"Bearer {key}"})
    except (requests.ConnectionError, requests.Timeout) as exc:
        raise Transient(f"network: {type(exc).__name__}") from exc
    seconds = time.perf_counter() - t0
    if r.status_code in (408, 429) or r.status_code >= 500:
        raise Transient(f"HTTP {r.status_code}")
    r.raise_for_status()                # 401/403/422 etc. stop the run: fix the key or payload
    data = r.json()
    a = data["answers"]
    return seconds, {
        "broad_type": a["broad_type"]["choice"],
        "cell_type": a["cell_type"]["choice"],
        "p_cell_type": a["cell_type"]["probabilities"][a["cell_type"]["choice"]],
        "coherence": a["identity_coherence"]["score"],
        "model_returned": data.get("model"),
    }, data

In [ ]:
client = anthropic.Anthropic(max_retries=0, timeout=300)  # retries handled by timed()
print("Claude endpoint:", client.base_url)


def answer_schema(questions):
    """Choice -> one of its option keys; score -> a level index."""
    props = {
        q: {"type": "string", "enum": list(spec["criteria"])} if spec["type"] == "choice"
        else {"type": "integer", "enum": list(range(len(spec["criteria"])))}
        for q, spec in questions.items()
    }
    return {"type": "object", "properties": props, "required": list(props), "additionalProperties": False}


SCHEMA = answer_schema(questions)
SYSTEM = (
    "You annotate clusters from single-cell RNA-seq data. You receive a cluster's state "
    "(marker genes ranked by differential expression against all other clusters, and the tissue) "
    "and a set of questions. Answer every question. For a choice question, answer with one of its "
    "option keys exactly. For a score question, answer with the index of the level that fits best, "
    "where 0 is the first level listed."
)


def ask_claude(model, state):
    prompt = (f"<state>\n{json.dumps(state, indent=2)}\n</state>\n\n"
              f"<questions>\n{json.dumps(questions, indent=2)}\n</questions>")
    t0 = time.perf_counter()
    try:
        response = client.messages.create(
            model=model,
            max_tokens=16000,
            system=SYSTEM,
            output_config={"effort": CLAUDE_EFFORT,
                           "format": {"type": "json_schema", "schema": SCHEMA}},
            messages=[{"role": "user", "content": prompt}],
        )
    except (anthropic.RateLimitError, anthropic.OverloadedError, anthropic.InternalServerError,
            anthropic.ServiceUnavailableError, anthropic.APIConnectionError) as exc:
        raise Transient(type(exc).__name__) from exc
    seconds = time.perf_counter() - t0
    answer = {"model_returned": response.model, "stop_reason": response.stop_reason}
    if response.stop_reason == "end_turn":
        a = json.loads(next(b.text for b in response.content if b.type == "text"))
        answer |= {"broad_type": a["broad_type"], "cell_type": a["cell_type"],
                   "coherence": a["identity_coherence"]}
    return seconds, answer, response.to_dict()

## Run

Each repeat runs every case through every model, so the run makes 3 models × 2 cases × `N_REPEATS` calls. The first call to each service includes connection setup, which is why the summary reports the median.

In [ ]:
route, jev_key = find_jev_route()
print(f"Jev route: {route} -> {JEV_ROUTES[route]['model']}")

records, raw = [], []
with requests.Session() as session:
    contenders = {"Jev": lambda s: ask_jev(s, session, route, jev_key)}
    contenders |= {name: (lambda s, m=model: ask_claude(m, s)) for name, model in CLAUDE_MODELS.items()}
    for repeat in range(1, N_REPEATS + 1):
        for case, s in cases.items():
            for name, ask in contenders.items():
                row, response = timed(lambda: ask(s))
                records.append({"case": case, "model": name, "repeat": repeat, **row})
                raw.append({"case": case, "model": name, "repeat": repeat, "response": response})
                print(f"rep {repeat}  {case:12s} {name:18s} {row.get('seconds', '-'):>7}s  "
                      f"{row.get('cell_type') or row.get('error') or row.get('stop_reason')}")

## Results: every call

In [ ]:
calls = pd.DataFrame(records)
cols = ["case", "model", "repeat", "status", "seconds", "broad_type", "cell_type", "p_cell_type", "coherence"]
calls[[c for c in cols if c in calls]]

## Summary: time and answers per case and model

- `cell_type` and `broad_type` show the most common answer and how many repeats gave it.
- `x_vs_jev` is the model's median time divided by Jev's median time for the same case.

In [ ]:
def modal(values):
    values = [v for v in values if v is not None]
    if not values:
        return None
    label, n = Counter(values).most_common(1)[0]
    return f"{label} ({n}/{len(values)})"


ok = calls[calls["status"] == "ok"]
summary = ok.groupby(["case", "model"], sort=False).agg(
    median_s=("seconds", "median"), min_s=("seconds", "min"), max_s=("seconds", "max"),
    cell_type=("cell_type", modal), broad_type=("broad_type", modal), coherence=("coherence", "mean"),
)
jev_median = summary.xs("Jev", level="model")["median_s"]
summary["x_vs_jev"] = (summary["median_s"] / summary.index.get_level_values("case").map(jev_median)).round(1)
summary.round(3)

## Save

Writes three files, stamped with the run time, to `outputs/`:
- every call;
- the summary;
- the raw responses.

Raw responses contain no credentials.

In [ ]:
OUTPUTS.mkdir(exist_ok=True)
stamp = time.strftime("%Y%m%d-%H%M%S")
calls.to_csv(OUTPUTS / f"calls_{stamp}.csv", index=False)
summary.to_csv(OUTPUTS / f"summary_{stamp}.csv")
(OUTPUTS / f"raw_{stamp}.json").write_text(json.dumps(raw, indent=2, default=str))
print("saved", sorted(p.name for p in OUTPUTS.glob(f"*_{stamp}.*")))

## What to look at

- **Does LST1 change any answer?** LST1 is a monocyte gene, so the list becomes slightly mixed.
- **Coherence**: does Jev's coherence drop in the `+ LST1` case? In the JevCellType demo it fell from 3.67 to 3.14 when LST1 and other genes were added. Do Claude's levels move too?
- **Agreement across repeats**: identical inputs can still give different answers. Jev's probabilities can drift slightly between calls; Claude can change its answer.
- **Time**: compare the `x_vs_jev` column.